# Power ADC Measurement Map

ADC readout gives us arbitrary unit of data. So, we have to map the ADC data to actual input power data. Note that for 200 MHz, it is recommended to use DC in, and above 500 MHz, it is recommended to use RF in, since RF in balun blocks the lower frequency.

## Instrument Instantiation

In [113]:
import numpy as np
import math
import matplotlib.pyplot as plt
import time
import pyvisa
from typing import Union, Any
from pprint import pprint

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qcodes.dataset.data_set import load_by_run_spec
from qick import *
from qick.averager_program import QickSweep
from qick.pyro import make_proxy

# Qick version : 0.2.357
(soc, soccfg) = make_proxy("192.168.2.99")

# Set DAC Channel 0 attenuation 20 dB and 20 dB, and turn on DAC channel
soc.rfb_set_gen_rf(0,0,0)
# Set DAC Channel filter as bypass mode
soc.rfb_set_gen_filter(0,fc = 2.5, ftype = "lowpass")

# Set ADC Channel attenuation 20 dB, and turn on ADC channel
soc.rfb_set_ro_rf(0,0)
# Set ADC Channel filter as bypass mode
soc.rfb_set_ro_filter(0, fc = 2.5, ftype = "lowpass")

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_8fd72669100a4219ac4728a5316556f8@192.168.2.99:41081


## QCodes Initialization

In [142]:
station = Station()

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251127_QICK_Test/gain_pwr_calb.db")
exp = load_or_create_experiment("2D sweep", "DC_In_200MHz")
meas = Measurement(exp=exp, station=station)

meas_freq = Parameter(name = "freq", label = "freq", unit = "Hz")
meas_val = Parameter(name = "measured_value", label = "measured_value", unit = "20 log AU")
meas_in_pwr = Parameter(name = "meas_in_pwr", label = "meas_in_pwr", unit = "dBm")
meas_slope = Parameter(name = "meas_slope", label = "meas_slope", unit = "dBm/20 log AU")
meas_intercept = Parameter(name = "meas_intercept", label = "meas_intercept", unit = "dBm")

meas.register_parameter(meas_val)
meas.register_parameter(meas_freq)

meas.register_parameter(meas_in_pwr, setpoints=(meas_freq, meas_val))
meas.register_parameter(meas_slope, setpoints=(meas_freq,))
meas.register_parameter(meas_intercept, setpoints=(meas_freq,))

## Measurement Program

In [110]:
class Power_ADC_Value_Mapping_Program(NDAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        freq_rf     = cfg["freq_rf"]
        # Declare RF generation channel
        self.declare_gen(
            ch      = 0,        # Channel
            nqz     = 1         # Nyquist Zone
        )
        # Declare RF input channel
        self.declare_readout(
            ch      = 2,        # Channel
            length  = int(cfg["pulse_time"] * 3/4) - 10,    # Readout length
                                                            # 10 is subtracted to
                                                            # make margin in timing
        )
        self.r_gain  = self.get_gen_reg(gen_ch = 0, name = "gain")
        self.add_sweep(
            QickSweep(
                prog    = self,
                reg     = self.r_gain,
                start   = cfg["start"],
                stop    = 32767,
                expts   = cfg["expts"],
                label   = "gain_sweep"
            )
        )

        # Convert RF frequency to DAC DDS register value
        freq_dac    = self.freq2reg(
            f       = freq_rf,  # Frequency
            gen_ch  = 0,        # Generator channel
            ro_ch   = 0         # Readout channel for round up
        )
        # Convert RF frequency to ADC DDS register value
        freq_adc    = self.freq2reg_adc(
            f       = freq_rf,  # Frequency
            ro_ch   = 2,        # Readout channel
            gen_ch  = 0         # Generator channel for round up
        )

        # Set DAC DDS
        self.set_pulse_registers(
            ch      = 0,        # Generator channel
            style   = "const",  # Output is gain * DDS output
            freq    = freq_dac, # Generator DDS frequency
            phase   = 0,        # Generator DDS phase
            gain    = 0,        # Generator amplitude
            length  = self.cfg["pulse_time"], # Pulse length
            phrst   = 0,        # Generator DDS phase reset
            mode    = "periodic"
        )
        # Set ADC DDS
        self.set_readout_registers(
            ch      = 2,        # Readout channel
            freq    = freq_adc, # Readout DDS frequency
            length  = self.cfg["pulse_time"], # Readout DDS multiplication length
            phrst   = 0         # Readout DDS phase reset
        )
        self.synci(100)

    def body(self):
        cfg = self.cfg
        self.pulse(
            ch      = 0,        # Generator channel
            t       = 100
        )
        self.readout(
            ch      = 2,        # Readout channel
            t       = 100       # Readout DDS will start multiplication
                                # @ sync_t + 100
        )
        # Make measurement triggers and shift t_sync
        for i in range(cfg["number_of_pulse"]):
            self.sync_all()
            self.trigger(
                adcs    = [2],      # Readout channels
                adc_trig_offset = 150 # Readout will capture the data @ sync_t + 50
            )

        self.sync_all(1000)
        # Make sure that do not read buffer before experiment ends
        self.wait_all()


## Plot Results

In [144]:
import json
id = 6

att1 = 31
att2 = 31
with meas.run() as datasaver:
    freqs = [180, 200, 220]
        
    datasaver.dataset.add_metadata(
        tag = f"Calibration_Result",
        metadata = json.dumps(
            {
                "freq" : freqs,
                "att1" : att1,
                "att2" : att2,
                "pwr_id" : id,
                "out_ch" : 0,
                "in_ch" : 2
            }
        )
    )
    for freq in freqs:
        cfg = {
            # Experiment Setup
            "reps" : 1,
            "expts" : 320,
            "start" : 1,
            "step" : 100,
            "freq_rf" : freq,
            # Parameter Setup
            "pulse_time" : 65000,
            "number_of_pulse" : 100,
        }
        prog = Power_ADC_Value_Mapping_Program(
            soccfg,
            cfg
        )

        soc.rfb_set_gen_rf(0,att1,att2)
        expts, avgi, avgq  = prog.acquire(soc = soc, progress = True, start_src = "internal")
        avgi = np.array(avgi[0]).mean(axis = 0)
        avgq = np.array(avgq[0]).mean(axis = 0)

        measured_power = 10 * np.log10(avgi * avgi + avgq * avgq)

        data = {}
        dataset = load_by_run_spec(captured_run_id=id)

        out_gain = dataset.get_parameter_data("gain")["gain"]["gain"]
        pwr = dataset.get_parameter_data("pwr")["pwr"]["pwr"]
        measured_freq = dataset.get_parameter_data("freq")["freq"]["freq"]

        for x in set(measured_freq):
            data[x] = {
                "out_gain": [],
                "pwr": [],
            }
        for index, x in enumerate(measured_freq):
            data[x]["out_gain"].append(out_gain[index])
            data[x]["pwr"].append(pwr[index])

        for x in set(measured_freq):
            if abs(x - freq) < 1:
                target_freq = x
                break
        if abs(target_freq - freq) > 1.5:
            raise ValueError(f"There is no power calibration data for {freq} MHz in {id}")
        slope_in, intercept_in = np.polyfit(np.log10(data[target_freq]["out_gain"][:-5]), data[target_freq]["pwr"][:-5], 1)
        slope, intercept = np.polyfit(measured_power, slope_in * np.log10(expts[0]) + intercept_in - att1 - att2, 1)
        print(f"pwr = {slope} * 10 log(ADC data) {intercept} [dBm] for freq {freq} MHz")
        datasaver.add_result(
            (meas_freq, [freq] * len(measured_power)),
            (meas_val, measured_power),
            (meas_in_pwr, np.array(slope_in * np.log10(expts[0]) + intercept_in - att1 - att2))
        )
        datasaver.add_result(
            (meas_freq, freq),
            (meas_slope, slope)
        )
        datasaver.add_result(
            (meas_freq, freq),
            (meas_intercept, intercept)
        )

Starting experimental run with id: 13. 


  0%|          | 0/320 [00:00<?, ?it/s]

pwr = 1.0670355549697383 * 10 log(ADC data) -65.67690348947717 [dBm] for freq 180 MHz


  0%|          | 0/320 [00:00<?, ?it/s]

pwr = 1.0586250806483783 * 10 log(ADC data) -65.70215820687145 [dBm] for freq 200 MHz


  0%|          | 0/320 [00:00<?, ?it/s]

pwr = 1.0562617162630508 * 10 log(ADC data) -64.88429022195258 [dBm] for freq 220 MHz
